[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fiit-ba/zneus-2026/blob/main/labs/week_03_backprop/task_2_backprop.ipynb)

# Week 3 · How a network calculates gradients

**Goal:** follow the chain rule from a loss back to every weight in week 2's **2 → 3 → 1** network. A gradient tells us how a small parameter change affects the loss; week 4 uses it to update the parameters.

Complete four **TODO** cells and three questions. All forward code and checks are supplied. Use CPU locally or in Colab; no downloads, accounts or hand-in are required. Keep your notebook for the test. In the Colab URL, replace `fiit-ba` with your GitHub username to open your fork.

*Adapted from the FIIT NSIETE course materials (vgg-fiit/NSIETE_2026).*

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

SEED = 42
DTYPE = torch.float64

def check(name, actual, expected, atol=1e-10):
    torch.testing.assert_close(actual, expected, atol=atol, rtol=0)
    print("OK:", name)

## 1. The chain rule through one neuron

Start with a linear neuron and squared error: $p=wx+b$, $L=(p-y)^2$.

$$\frac{\partial L}{\partial p}=2(p-y),\qquad
\frac{\partial L}{\partial w}=\frac{\partial L}{\partial p}\,x,\qquad
\frac{\partial L}{\partial b}=\frac{\partial L}{\partial p}.$$

For $x=2$, $w=0.5$, $b=0$, $y=0$, the prediction is $1$, loss is $1$, and the weight gradient is $4$. A small increase in the weight increases the loss. Run this worked example:

In [ ]:
x, w, b, target = 2., 0.5, 0., 0.
prediction = w * x + b
loss = (prediction - target) ** 2
d_prediction = 2 * (prediction - target)
d_w = d_prediction * x
d_b = d_prediction
print(f"prediction={prediction}, loss={loss}, d_w={d_w}, d_b={d_b}")

## 2. Start at the loss

For a batch of $m$ samples with one output each, use **mean squared error**:

$$J=\frac{1}{m}\sum_i(p_i-y_i)^2,\qquad
\frac{\partial J}{\partial p_i}=\frac{2(p_i-y_i)}{m}.$$

**TODO 1:** return that gradient. `prediction.numel()` counts the outputs ($m$ here). We divide **once, here**; every later `backward` receives a gradient of the same mean loss $J$ and must not divide again.

MSE keeps the first derivation simple. Binary cross-entropy is shown in the optional section.

In [ ]:
class MSELoss:
    def forward(self, prediction, target):
        return ((prediction - target) ** 2).mean()

    def backward(self, prediction, target):
        return ...  # TODO 1: derivative of the mean squared error

In [ ]:
criterion = MSELoss()
p = torch.tensor([[0.2, 0.8]], dtype=DTYPE)
y = torch.tensor([[0., 1.]], dtype=DTYPE)
p_ref = p.clone().requires_grad_(True)
F.mse_loss(p_ref, y).backward()
check("loss value", criterion.forward(p, y), F.mse_loss(p, y))
check("loss gradient", criterion.backward(p, y), p_ref.grad)

## 3. Back through a linear layer

Keep week 2's column convention: `X: (inputs, samples)`, `W: (outputs, inputs)`, `b: (outputs, 1)`.
The forward pass saves `X`, because backward needs it. For $Z=WX+b$:

$$dW=dZ\,X^T,\qquad db=\sum_{\text{samples}}dZ,\qquad dX=W^T dZ.$$

Here `dZ` means $\partial J/\partial Z$. We sum bias contributions because the same bias was used for every sample. **TODO 2:** store `dW`, store `db` with `sum(dim=1, keepdim=True)`, and return `dX`. This calculates gradients; it does not change the weights.

In [ ]:
class Module:
    def __init__(self):
        self.modules = {}

    def add_module(self, module, name):
        self.modules[name] = module

    def __call__(self, X):
        return self.forward(X)

class Linear(Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.W = torch.randn(out_features, in_features, dtype=DTYPE) * 0.1
        self.b = torch.zeros(out_features, 1, dtype=DTYPE)
        self.dW = torch.zeros_like(self.W)
        self.db = torch.zeros_like(self.b)

    def forward(self, X):
        self.X = X
        return self.W @ X + self.b

    def backward(self, dZ):
        self.dW = ...  # TODO 2: dZ @ X.T
        self.db = ...  # TODO 2: sum over sample columns
        return ...  # TODO 2: W.T @ dZ

In [ ]:
torch.manual_seed(SEED)
layer = Linear(2, 3)
inputs = torch.randn(2, 4, dtype=DTYPE)
dZ = torch.randn(3, 4, dtype=DTYPE)
layer(inputs)
dX = layer.backward(dZ)
W, b, inputs_ref = [t.clone().requires_grad_(True) for t in (layer.W, layer.b, inputs)]
((W @ inputs_ref + b) * dZ).sum().backward()
check("linear dW", layer.dW, W.grad)
check("linear db", layer.db, b.grad)
check("linear dX", dX, inputs_ref.grad)

## 4. Back through an activation

If $A=g(Z)$, the chain rule gives **$dZ=dA\,g'(Z)$**, element by element.

- **ReLU:** its derivative is $1$ for $Z>0$ and $0$ for $Z<0$. We choose $0$ at the corner $Z=0$.
- **Sigmoid:** its derivative is $A(1-A)$, so saving its forward output is enough.

**TODO 3:** implement ReLU's backward pass with `dA * (self.X > 0)`. Sigmoid is supplied; trace the same chain-rule multiplication in its code.

In [ ]:
class ReLU(Module):
    def forward(self, X):
        self.X = X
        return torch.clamp(X, min=0)

    def backward(self, dA):
        return ...  # TODO 3: pass the gradient only where X > 0

class Sigmoid(Module):
    def forward(self, X):
        self.A = 1 / (1 + torch.exp(-X))
        return self.A

    def backward(self, dA):
        return dA * self.A * (1 - self.A)

In [ ]:
inputs = torch.tensor([[-2., 0., 2.]], dtype=DTYPE)
dA = torch.tensor([[2., 3., 4.]], dtype=DTYPE)
for activation, reference in [(ReLU(), torch.relu), (Sigmoid(), torch.sigmoid)]:
    activation(inputs)
    inputs_ref = inputs.clone().requires_grad_(True)
    (reference(inputs_ref) * dA).sum().backward()
    check(type(activation).__name__, activation.backward(dA), inputs_ref.grad)

## 5. Follow the whole chain

Forward goes `Linear → ReLU → Linear → Sigmoid`; backward goes in the **reverse order**. Each layer returns the gradient needed by the layer before it.

**TODO 4:** fill `dA = layer.backward(dA)` inside the reversed loop. The supplied `parameters()` returns the weight/gradient and bias/gradient pairs for our checks and next week's optimizers.

In [ ]:
class Model(Module):
    def forward(self, X):
        for layer in self.modules.values():
            X = layer(X)
        return X

    def backward(self, dA):
        for layer in reversed(self.modules.values()):
            dA = ...  # TODO 4: use this layer's backward pass
        return dA

    def parameters(self):
        pairs = []
        for layer in self.modules.values():
            if isinstance(layer, Linear):
                pairs.extend([(layer.W, layer.dW), (layer.b, layer.db)])
        return pairs

**Given example for weeks 3 and 4: XOR.** Two inputs have target `1` when their signs differ and `0` when they match. The two classes lie on opposite diagonals, so one straight line cannot separate them. We keep the **2 → 3 → 1** architecture from week 2.

In [ ]:
X = torch.tensor([[-1., -1., 1., 1.], [-1., 1., -1., 1.]], dtype=DTYPE)
Y = torch.tensor([[0., 1., 1., 0.]], dtype=DTYPE)
plt.scatter(X[0], X[1], c=Y[0], cmap="coolwarm", vmin=0, vmax=1, s=100)
plt.xlabel("input 1")
plt.ylabel("input 2")
plt.title("XOR: blue = 0, red = 1")
plt.show()

In [ ]:
def create_model():
    torch.manual_seed(SEED)
    model = Model()
    model.add_module(Linear(2, 3), "hidden")
    model.add_module(ReLU(), "relu")
    model.add_module(Linear(3, 1), "output")
    model.add_module(Sigmoid(), "sigmoid")
    return model

In [ ]:
model = create_model()
prediction = model(X)
loss = criterion.forward(prediction, Y)
d_prediction = criterion.backward(prediction, Y)
d_inputs = model.backward(d_prediction)
print("predictions:", prediction)
print("mean loss:", loss.item())

### Check the gradients (given)

The reference below uses PyTorch autograd on copies of the same weights and inputs. Our network calculates its own gradients without autograd. Each `OK` checks a different part of the chain.

In [ ]:
reference = [p.clone().requires_grad_(True) for p, gradient in model.parameters()]
W1, b1, W2, b2 = reference
inputs_ref = X.clone().requires_grad_(True)
p_ref = torch.sigmoid(W2 @ torch.relu(W1 @ inputs_ref + b1) + b2)
F.mse_loss(p_ref, Y).backward()
check("network output", prediction, p_ref.detach())
check("network input gradient", d_inputs, inputs_ref.grad)
for name, (parameter, gradient), ref in zip(["hidden W", "hidden b", "output W", "output b"], model.parameters(), reference):
    check(name, gradient, ref.grad)

**A second check: change a weight and observe the loss.** For each parameter $\theta$, central differences estimate its gradient:

$$\frac{\partial J}{\partial\theta}\approx\frac{J(\theta+\varepsilon)-J(\theta-\varepsilon)}{2\varepsilon}.$$

Run the supplied check below. It restores each parameter after measuring. We use `float64` because subtracting nearly equal losses loses precision; this example stays away from ReLU's corner.

In [ ]:
epsilon = 1e-6
for parameter, gradient in model.parameters():
    numeric = torch.zeros_like(parameter)
    for i in range(parameter.numel()):
        original = parameter.view(-1)[i].item()
        parameter.view(-1)[i] = original + epsilon
        plus = criterion.forward(model(X), Y)
        parameter.view(-1)[i] = original - epsilon
        minus = criterion.forward(model(X), Y)
        parameter.view(-1)[i] = original
        numeric.view(-1)[i] = (plus - minus) / (2 * epsilon)
    check("finite differences", gradient, numeric, atol=1e-8)
model(X)  # restore the forward caches after the perturbations
print("All required gradient checks passed.")

## 6. Ready for the test?

1. Why does backward visit layers in reverse order, and why must forward run first?

   _Your answer here._

2. Where do we divide by the batch size? What would happen if every linear layer divided again?

   _Your answer here._

3. What does a positive weight gradient tell you? How does the finite-difference check test that meaning?

   _Your answer here._

Restart and run all cells: each check should print `OK`. Practise tracing the scalar example and the network's gradient shapes. **No notebook or link submission is required.**

## Optional: other activations and losses

The same chain rule works for Tanh, with derivative $1-A^2$, and LeakyReLU, with derivative $1$ on positive inputs and slope $\alpha$ on negative inputs.

For binary labels and probabilities $0<p<1$, **binary cross-entropy** penalizes confident wrong predictions more strongly:

$$J=-\frac{1}{m}\sum_i[y_i\log p_i+(1-y_i)\log(1-p_i)],\qquad
\frac{\partial J}{\partial p_i}=\frac{-y_i/p_i+(1-y_i)/(1-p_i)}{m}.$$

The supplied example uses probabilities strictly between 0 and 1. Run it and compare its gradient with MSE's for the same predictions. In later PyTorch models, `BCEWithLogitsLoss` combines sigmoid and BCE for numerical stability.

In [ ]:
class BCELoss:
    def forward(self, p, y):
        return -(y * p.log() + (1 - y) * (1 - p).log()).mean()

    def backward(self, p, y):
        return (-y / p + (1 - y) / (1 - p)) / p.numel()

bce = BCELoss()
p_ref = p.clone().requires_grad_(True)
F.binary_cross_entropy(p_ref, y).backward()
check("BCE value", bce.forward(p, y), F.binary_cross_entropy(p, y))
check("BCE gradient", bce.backward(p, y), p_ref.grad)
print("MSE gradient:", criterion.backward(p, y), "BCE gradient:", bce.backward(p, y))